In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("prathumarikeri/indian-sign-language-isl")

print("Path to dataset files:", path)

100%|██████████| 281M/281M [00:04<00:00, 69.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/prathumarikeri/indian-sign-language-isl/versions/1


In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("prathumarikeri/indian-sign-language-isl")

print("Path to dataset files:", path)

"""
train_mobilenetv2_colab.py — MobileNetV2 ISL Trainer (Google Colab version)
Just edit the CONFIG section below and run all cells.
"""

# ╔══════════════════════════════════════════════════════╗
# ║              ✏️  EDIT ONLY THIS SECTION              ║
# ╚══════════════════════════════════════════════════════╝

CONFIG = {
    # Path to your dataset folder (subfolders = class names A, B, C ...)
    # If using Kaggle in Colab, typically: "/kaggle/input/indian-sign-language-isl"
    # If uploaded to Drive: "/content/drive/MyDrive/ISL_dataset"
    "dataset"      : os.path.join(path, "Indian"),

    # Where to save the output model and labels
    "output"       : "/content/models",

    # Training settings
    "epochs"       : 20,
    "img_size"     : 128,       # must stay 128 — matches your app
    "batch_size"   : 32,
    "val_split"    : 0.2,

    # Fine-tuning: unfreeze MobileNetV2 layers from this index onwards
    # Lower = more layers unfrozen = slower but potentially more accurate
    "fine_tune_at" : 100,
}

# ╔══════════════════════════════════════════════════════╗
# ║         DO NOT EDIT BELOW THIS LINE                  ║
# ╚══════════════════════════════════════════════════════╝

import sys
import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# ── Validate dataset ──────────────────────────────────────────────────────────
dataset_path = CONFIG["dataset"]

if not os.path.isdir(dataset_path):
    raise FileNotFoundError(
        f"\n[ERROR] Dataset folder not found: {dataset_path}\n"
        "Please update CONFIG['dataset'] with the correct path."
    )

classes = sorted([
    d for d in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, d))
])

if not classes:
    raise ValueError(f"[ERROR] No subfolders found in {dataset_path}")

num_classes = len(classes)
IMG_SIZE    = CONFIG["img_size"]
BATCH_SIZE  = CONFIG["batch_size"]
EPOCHS      = CONFIG["epochs"]
output_dir  = CONFIG["output"]

os.makedirs(output_dir, exist_ok=True)

print("=" * 55)
print("  ISL MobileNetV2 Trainer — Google Colab")
print("=" * 55)
print(f"  Dataset    : {dataset_path}")
print(f"  Classes    : {num_classes}  → {classes}")
print(f"  Image size : {IMG_SIZE}x{IMG_SIZE} grayscale")
print(f"  Epochs     : {EPOCHS}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Output     : {output_dir}")
print("=" * 55)

# ── TensorFlow imports ────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)

print(f"\nTensorFlow : {tf.__version__}")
print(f"GPUs       : {tf.config.list_physical_devices('GPU')}\n")

# ── Data generators ───────────────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=CONFIG["val_split"],
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=False,      # ISL signs are hand-specific — no flip
    brightness_range=[0.7, 1.3],
    fill_mode="nearest",
)

train_gen = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=42,
)

val_gen = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode="grayscale",
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=42,
)

print(f"Training samples  : {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Class mapping     : {train_gen.class_indices}\n")

# ── Save labels.json ──────────────────────────────────────────────────────────
label_map   = {str(v): k for k, v in train_gen.class_indices.items()}
labels_path = os.path.join(output_dir, "labels.json")
with open(labels_path, "w") as f:
    json.dump(label_map, f, indent=2)
print(f"Labels saved → {labels_path}")

# ── Build model ───────────────────────────────────────────────────────────────
# App sends (1,128,128,1) grayscale → Lambda converts to RGB inside model
# Your app's preprocessor needs ZERO changes

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="grayscale_input")

x = layers.Lambda(
    lambda t: tf.repeat(t, 3, axis=-1),
    name="gray_to_rgb"
)(inputs)

x = layers.Lambda(
    lambda t: (t * 2.0) - 1.0,
    name="mobilenet_preprocess"
)(x)

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dense(256, activation="relu", name="fc1")(x)
x = layers.Dropout(0.4, name="drop1")(x)
x = layers.Dense(128, activation="relu", name="fc2")(x)
x = layers.Dropout(0.3, name="drop2")(x)
outputs = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

model = keras.Model(inputs, outputs, name="ISL_MobileNetV2")
model.summary()

# ── Phase 1 — Train head only (backbone frozen) ───────────────────────────────
print("\n" + "="*55)
print("  PHASE 1 — Head training (backbone frozen)")
print("="*55)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

phase1_epochs = min(10, EPOCHS // 2)

history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=phase1_epochs,
    callbacks=[
        EarlyStopping(monitor="val_accuracy", patience=5,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=3, verbose=1),
        CSVLogger(os.path.join(output_dir, "phase1_log.csv")),
    ],
    verbose=1,
)
print(f"\nPhase 1 best val_accuracy: {max(history1.history['val_accuracy']):.4f}")

# ── Phase 2 — Fine-tune top backbone layers ───────────────────────────────────
print("\n" + "="*55)
print(f"  PHASE 2 — Fine-tuning from layer {CONFIG['fine_tune_at']}")
print("="*55)

base_model.trainable = True
for layer in base_model.layers[:CONFIG["fine_tune_at"]]:
    layer.trainable = False

print(f"  Trainable backbone layers: "
      f"{sum(1 for l in base_model.layers if l.trainable)} / {len(base_model.layers)}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

best_model_path = os.path.join(output_dir, "isl_model.h5")
phase2_epochs   = EPOCHS - phase1_epochs

history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=phase2_epochs,
    callbacks=[
        ModelCheckpoint(best_model_path, monitor="val_accuracy",
                        save_best_only=True, verbose=1),
        EarlyStopping(monitor="val_accuracy", patience=7,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.3,
                          patience=3, min_lr=1e-6, verbose=1),
        CSVLogger(os.path.join(output_dir, "phase2_log.csv")),
    ],
    verbose=1,
)
print(f"\nPhase 2 best val_accuracy: {max(history2.history['val_accuracy']):.4f}")

# ── Save final model ──────────────────────────────────────────────────────────
model.save(best_model_path)
print(f"\nModel saved → {best_model_path}")

# ── Save model_info.json ──────────────────────────────────────────────────────
info = {
    "model_name"    : "ISL_MobileNetV2",
    "architecture"  : "MobileNetV2 + custom head",
    "input_shape"   : [IMG_SIZE, IMG_SIZE, 1],
    "input_type"    : "grayscale_image",
    "num_classes"   : num_classes,
    "classes"       : classes,
    "preprocessing" : {
        "color_mode": "grayscale",
        "resize"    : [IMG_SIZE, IMG_SIZE],
        "normalize" : "divide_by_255"
    },
    "notes": "Grayscale→RGB handled inside model. App preprocessor unchanged."
}
with open(os.path.join(output_dir, "model_info.json"), "w") as f:
    json.dump(info, f, indent=2)

# ── Plot training history ─────────────────────────────────────────────────────
acc   = history1.history["accuracy"]     + history2.history["accuracy"]
val   = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss  = history1.history["loss"]         + history2.history["loss"]
vloss = history1.history["val_loss"]     + history2.history["val_loss"]
ep    = range(1, len(acc) + 1)
pb    = len(history1.history["accuracy"])   # phase boundary

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("ISL MobileNetV2 — Training History")
ax1.plot(ep, acc,  "b-", label="Train"); ax1.plot(ep, val,  "r-", label="Val")
ax1.axvline(pb, color="gray", linestyle="--", label="Fine-tune")
ax1.set_title("Accuracy"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(ep, loss, "b-", label="Train"); ax2.plot(ep, vloss,"r-", label="Val")
ax2.axvline(pb, color="gray", linestyle="--", label="Fine-tune")
ax2.set_title("Loss"); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plot_path = os.path.join(output_dir, "training_history.png")
plt.savefig(plot_path, dpi=120); plt.close()
print(f"Plot saved → {plot_path}")

# ── Final evaluation ──────────────────────────────────────────────────────────
print("\n" + "="*55)
print("  FINAL EVALUATION")
print("="*55)
val_gen.reset()
loss_val, acc_val = model.evaluate(val_gen, verbose=1)
print(f"\n  Val Accuracy : {acc_val*100:.2f}%")
print(f"\n{'='*55}")
print(f"  ✅ Done! Download these files to your PC:")
print(f"     {best_model_path}")
print(f"     {labels_path}")
print(f"  Then copy both into ISL_Translator/models/")
print(f"{'='*55}")


Using Colab cache for faster access to the 'indian-sign-language-isl' dataset.
Path to dataset files: /kaggle/input/indian-sign-language-isl
  ISL MobileNetV2 Trainer — Google Colab
  Dataset    : /kaggle/input/indian-sign-language-isl/Indian
  Classes    : 35  → ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
  Image size : 128x128 grayscale
  Epochs     : 20
  Batch size : 32
  Output     : /content/models

TensorFlow : 2.20.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Found 34198 images belonging to 35 classes.
Found 8547 images belonging to 35 classes.
Training samples  : 34198
Validation samples: 8547
Class mapping     : {'1': 0, '2': 1, '3': 2, '4': 3, '5': 4, '6': 5, '7': 6, '8': 7, '9': 8, 'A': 9, 'B': 10, 'C': 11, 'D': 12, 'E': 13, 'F': 14, 'G': 15, 'H': 16, 'I': 17, 'J': 18, 'K': 19, 'L': 20, 'M': 21, 'N': 22, 'O

Model: "ISL_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ grayscale_input (InputLayer)    │ (None, 128, 128, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gray_to_rgb (Lambda)            │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_preprocess (Lambda)   │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 35)             │         4,515 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,623,331 (10.01 MB)

 Trainable params: 365,347 (1.39 MB)

...


Epoch 1: finished saving model to /content/models/isl_model.h5
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 169s 136ms/step - accuracy: 0.9884 - loss: 0.0577 - val_accuracy: 0.9874 - val_loss: 0.1339 - learning_rate: 1.0000e-04
Epoch 2/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.9972 - loss: 0.0108
Epoch 2: val_accuracy improved from 0.98736 to 0.98830, saving model to /content/models/isl_model.h5



Epoch 2: finished saving model to /content/models/isl_model.h5
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 179s 126ms/step - accuracy: 0.9978 - loss: 0.0087 - val_accuracy: 0.9883 - val_loss: 0.2912 - learning_rate: 1.0000e-04
Epoch 3/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - accuracy: 0.9970 - loss: 0.0163
Epoch 3: val_accuracy did not improve from 0.98830
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 128s 119ms/step - accuracy: 0.9972 - loss: 0.0155 - val_accuracy: 0.9788 - val_loss: 0.2369 - learning_rate: 1.0000e-04
Epoch 4/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.9984 - loss: 0.0103
Epoch 4: val_accuracy improved from 0.98830 to 0.98865, saving model to /content/models/isl_model.h5



Epoch 4: finished saving model to /content/models/isl_model.h5

Epoch 4: ReduceLROnPlateau reducing learning rate to 2.9999999242136255e-05.
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 129s 121ms/step - accuracy: 0.9985 - loss: 0.0082 - val_accuracy: 0.9887 - val_loss: 0.1766 - learning_rate: 1.0000e-04
Epoch 5/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.9989 - loss: 0.0078
Epoch 5: val_accuracy improved from 0.98865 to 0.98947, saving model to /content/models/isl_model.h5



Epoch 5: finished saving model to /content/models/isl_model.h5
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 123s 115ms/step - accuracy: 0.9994 - loss: 0.0041 - val_accuracy: 0.9895 - val_loss: 0.1551 - learning_rate: 3.0000e-05
Epoch 6/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.9997 - loss: 0.0011
Epoch 6: val_accuracy did not improve from 0.98947
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 123s 115ms/step - accuracy: 0.9998 - loss: 9.3113e-04 - val_accuracy: 0.9895 - val_loss: 0.1331 - learning_rate: 3.0000e-05
Epoch 7/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - accuracy: 0.9997 - loss: 9.0197e-04
Epoch 7: val_accuracy did not improve from 0.98947
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 126s 118ms/step - accuracy: 0.9997 - loss: 0.0011 - val_accuracy: 0.9895 - val_loss: 0.1248 - learning_rate: 3.0000e-05
Epoch 8/10
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.9997 - loss: 5.6527e-04
Epoch 8: val_accuracy did not improve from 0.98947
1069/1069 ━━━━━━━━━━━━━━━━━━━━ 135s 126ms/step - a


Phase 2 best val_accuracy: 0.9895

Model saved → /content/models/isl_model.h5
Plot saved → /content/models/training_history.png

  FINAL EVALUATION
268/268 ━━━━━━━━━━━━━━━━━━━━ 26s 96ms/step - accuracy: 0.9895 - loss: 0.1522

  Val Accuracy : 98.95%

  ✅ Done! Download these files to your PC:
     /content/models/isl_model.h5
     /content/models/labels.json
  Then copy both into ISL_Translator/models/
